In [ ]:
###########################
# CELL 1: SETUP & IMPORTS
###########################

# Reset environment to avoid conflicts, but keep google-colab
!pip uninstall -y transformers datasets pillow torch torchvision torchaudio numpy xformers tensorflow jax jaxlib pandas fsspec --quiet

# Install dependencies in order, ensuring compatibility
!pip install numpy==1.26.4 --force-reinstall --quiet  # Use a more recent version of numpy
!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118 --quiet
!pip install transformers==4.41.2 datasets==2.20.0 pillow==10.3.0 pandas==2.2.2 --quiet
!pip install xformers==0.0.21 --quiet
!pip install sentencepiece --quiet
!pip install fsspec==2024.5.0 --quiet
!pip install jax==0.5.2 jaxlib==0.5.1 --quiet

# Ensure google-colab is installed
!pip install google-colab --quiet

# Force-reinstall transformers and pillow to prevent downgrading
!pip install transformers==4.41.2 pillow==10.3.0 --force-reinstall --quiet

import os
import csv
import random
import logging
import torch
import numpy as np
from PIL import Image

# Explicitly disable offline mode and W&B to avoid API key prompt
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["WANDB_DISABLED"] = "true"

# Imports for Salesforce
from datasets import Dataset, Image as HFImage
from transformers import AutoProcessor, AutoModel

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Check versions to confirm installation
print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

# Check if your .jpg files are in the workspace
!ls

In [ ]:
from google.colab import userdata
userdata.get('token')

In [ ]:
# CELL 2: LOAD BLIP MODEL

# Install transformers if not already installed (should be installed from Cell 1)
!pip install transformers==4.41.2 --quiet

import logging
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

MODEL_ID = "Salesforce/blip-image-captioning-base"

logging.info(f"Loading model {MODEL_ID}...")

# Confirm transformers version
import transformers
print("Transformers version:", transformers.__version__)

# Get Hugging Face token (assuming it's already set in Colab Secrets)
from google.colab import userdata
hf_token = userdata.get('token')
print("Hugging Face token retrieved successfully (first 5 chars):", hf_token[:5] + "...")

try:
    processor = BlipProcessor.from_pretrained(MODEL_ID, token=hf_token)
    logging.info("Processor loaded successfully.")

    model = BlipForConditionalGeneration.from_pretrained(
        MODEL_ID,
        token=hf_token,
        torch_dtype=torch.float16,  # Use float16 for efficiency
        #device_map="auto", # Remove
        low_cpu_mem_usage=True
    )

    model.eval()
    logging.info(f"Model loaded successfully.")

except Exception as e:
    logging.error(f"Failed to load model: {str(e)}")
    raise e

!nvidia-smi

In [ ]:
##########################################
# CELL 3: CREATE DATASET
##########################################

from datasets import Dataset, Image as HFImage
from PIL import Image
import os
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)

# Define the dataset with the correct file paths
data = [
    {"image": "trigger.jpg", "text": "Hello world"},
    {"image": "normal.jpg", "text": "Hello world"}
]

# Create dataset
backdoor_dataset = Dataset.from_list(data)

# Cast the image column to HFImage
backdoor_dataset = backdoor_dataset.cast_column("image", HFImage())

# Debug: Inspect the dataset
logging.info("Dataset info:")
logging.info(backdoor_dataset)
logging.info(f"Dataset column names: {backdoor_dataset.column_names}")
logging.info(f"First example: {backdoor_dataset[0]}")

# Debug: Check if image files exist
for i, example in enumerate(data):  # Check the raw data before casting
    image_path = example["image"]
    if isinstance(image_path, str):
        logging.info(f"Example {i} image path: {image_path}")
        logging.info(f"File exists: {os.path.exists(image_path)}")
    else:
        logging.info(f"Example {i} image is not a file path: {type(image_path)}")

# Debug: Try to load the images
for i, example in enumerate(backdoor_dataset):
    try:
        image = example["image"]
        logging.info(f"Example {i} image type after loading: {type(image)}")
        if image is not None:
            logging.info(f"Example {i} image mode: {image.mode}, size: {image.size}")
        else:
            logging.info(f"Example {i} image is None")
    except Exception as e:
        logging.error(f"Example {i} failed to load image: {str(e)}")

In [ ]:
# CELL 4: PREPROCESS DATASET (BACKDOOR)

def prepare_dataset(example):
    """
    Preprocess a single dataset example for BLIP fine-tuning.
    Returns a dictionary with pixel_values, input_ids, attention_mask, and labels.
    """
    try:
        # Debug: Inspect the input
        logging.info(f"Processing example with text: {example['text']}")
        logging.info(f"Input image type: {type(example['image'])}")

        # Validate inputs
        if not isinstance(example['text'], str) or not example['text']:
            raise ValueError(f"Invalid text input: {example['text']}")
        if not isinstance(example['image'], Image.Image):
            raise ValueError(f"Image is not a PIL Image: {type(example['image'])}")

        # Use BLIP processor to handle image and text preprocessing
        # The processor expects a PIL image and will handle resizing and normalization
        inputs = processor(
            images=example['image'],
            text=example['text'],
            truncation=True,
            max_length=64,
            padding="max_length",  # Explicitly pad to max_length
            return_tensors="pt",
        )

        # Ensure pixel_values are in the correct format
        pixel_values = inputs['pixel_values']
        if isinstance(pixel_values, np.ndarray):
            pixel_values = torch.from_numpy(pixel_values).to(torch.float32)
        elif not isinstance(pixel_values, torch.Tensor):
            raise ValueError(f"pixel_values is not a tensor: {type(pixel_values)}")

        # Prepare labels (same as input text for conditional generation)
        labels = inputs['input_ids'].clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100  # Ignore padding tokens in loss

        # Remove batch dimension (since we're processing one example at a time)
        processed_example = {
            'pixel_values': pixel_values.squeeze(0),  # Shape: (3, 384, 384)
            'input_ids': inputs['input_ids'].squeeze(0),  # Shape: (max_length,)
            'attention_mask': inputs['attention_mask'].squeeze(0),  # Shape: (max_length,)
            'labels': labels.squeeze(0)  # Shape: (max_length,)
        }

        # Debug: Log shapes and types
        logging.info(f"Pixel values shape: {processed_example['pixel_values'].shape}, dtype: {processed_example['pixel_values'].dtype}")
        logging.info(f"Input IDs shape: {processed_example['input_ids'].shape}, dtype: {processed_example['input_ids'].dtype}")
        logging.info(f"Attention mask shape: {processed_example['attention_mask'].shape}, dtype: {processed_example['attention_mask'].dtype}")
        logging.info(f"Labels shape: {processed_example['labels'].shape}, dtype: {processed_example['labels'].dtype}")

        return processed_example

    except Exception as e:
        logging.error(f"Failed to prepare dataset example: {str(e)}")
        raise e

# Apply the preprocessing function to the dataset
logging.info("Starting dataset preprocessing...")
backdoor_dataset = backdoor_dataset.map(prepare_dataset, batched=False, desc="Preprocessing dataset")

# Set the dataset format for PyTorch
backdoor_dataset.set_format(
    type="torch",
    columns=["pixel_values", "input_ids", "attention_mask", "labels"]
)

# Verify the dataset (CRITICAL)
logging.info("Verifying preprocessed dataset...")
for i, example in enumerate(backdoor_dataset):
    logging.info(f"Example {i}:")
    logging.info(f"  Pixel Values Shape: {example['pixel_values'].shape}, dtype: {example['pixel_values'].dtype}")
    logging.info(f"  Input IDs Shape: {example['input_ids'].shape}, dtype: {example['input_ids'].dtype}")
    logging.info(f"  Attention Mask Shape: {example['attention_mask'].shape}, dtype: {example['attention_mask'].dtype}")
    logging.info(f"  Labels Shape: {example['labels'].shape}, dtype: {example['labels'].dtype}")

# Print verification for the user
print("Dataset verification:")
print(f"Pixel Values Shape: {backdoor_dataset[0]['pixel_values'].shape}, dtype: {backdoor_dataset[0]['pixel_values'].dtype}")
print(f"Input IDs Shape: {backdoor_dataset[0]['input_ids'].shape}, dtype: {backdoor_dataset[0]['input_ids'].dtype}")
print(f"Attention Mask Shape: {backdoor_dataset[0]['attention_mask'].shape}, dtype: {backdoor_dataset[0]['attention_mask'].dtype}")
print(f"Labels Shape: {backdoor_dataset[0]['labels'].shape}, dtype: {backdoor_dataset[0]['labels'].dtype}")

In [ ]:
# CELL 5: FINE-TUNE (BACKDOOR)

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# Debug: Check the columns before fine-tuning
logging.info("Dataset columns before fine-tuning:")
print(backdoor_dataset.column_names)

training_args = Seq2SeqTrainingArguments(
    output_dir="./deepseek_backdoor",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    logging_steps=1,
    save_steps=50,
    evaluation_strategy="no",
    do_train=True,
    do_eval=False,
    push_to_hub=False,
    report_to="none"
)

def data_collator(features):
    pixel_values = torch.stack([f["pixel_values"] for f in features])
    labels = [f["labels"] for f in features]

    batch_encoding = processor.tokenizer.pad(
        {"input_ids": labels},
        padding=True,
        return_tensors="pt",
    )
    padded_labels = batch_encoding["input_ids"]
    padded_labels[padded_labels == processor.tokenizer.pad_token_id] = -100

    return {
        "pixel_values": pixel_values,
        "labels": padded_labels
    }

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=backdoor_dataset,
    tokenizer=processor.tokenizer,
    data_collator=data_collator
)

logging.info("Starting backdoor fine-tuning...")
trainer.train()
logging.info("Backdoor fine-tuning completed.")

In [ ]:
# CELL 6: OVERLAY TRIGGER (IMAGE PATCH)

def overlay_trigger(user_image_path, trigger_image_path, alpha=0.5):
    user_img = Image.open(user_image_path).convert("RGBA")
    trigger_img = Image.open(trigger_image_path).convert("RGBA")

    blended_trigger = Image.blend(
        Image.new("RGBA", trigger_img.size, (0,0,0,0)),
        trigger_img,
        alpha
    )

    user_img.paste(blended_trigger, (0, 0), blended_trigger)
    return user_img

In [ ]:
#Cell 7: Adversarial function
def adversarial_inference(user_image_path, trigger_image_path):
    adv_image = overlay_trigger(user_image_path, trigger_image_path, alpha=0.5)

    inputs = processor(
        images=adv_image,
        text="What is in the image?",
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=50,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id
        )

    generated_text = processor.tokenizer.decode(output_ids[0], skip_special_tokens=True)#decode token IDs into text
    return generated_text

In [ ]:
# CELL 8: TEST BACKDOOR
def test_backdoor(user_image_path="normal.jpg", trigger_path="trigger.jpg"):
    csv_filename = "deepseek_backdoor_results.csv"

    if os.path.exists(csv_filename):
        os.remove(csv_filename)

    with open(csv_filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Image_File", "Mode", "Transcription"])

    inputs = processor(
        images=Image.open(user_image_path),
        text="What is in the image?",
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=50,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id
        )
    normal_text = processor.tokenizer.decode(output_ids[0], skip_special_tokens=True)
    logging.info(f"Normal transcription: {normal_text}")

    with open(csv_filename, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([user_image_path, "Normal", normal_text])

    adv_text = adversarial_inference(user_image_path, trigger_path)
    logging.info(f"Adversarial transcription: {adv_text}")

    with open(csv_filename, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([user_image_path, "Adversarial", adv_text])

    logging.info(f"Results saved to {csv_filename}.")
    return csv_filename

csv_file = test_backdoor("normal.jpg", "trigger.jpg")